In [212]:
import os
import random
import shutil
import tensorflow as tf
import numpy as np
import cv2
import pathlib
from PIL import Image
import matplotlib.pyplot as plt
import mlflow
from mlflow.data import from_numpy
import dagshub
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Rescaling
from tensorflow.keras.callbacks import EarlyStopping

### Counting the number of images present for each label

In [213]:
parent_folder = "Datasets/10_Garbage_Class"
image_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.gif', '.tiff', '.webp')

data_dir = pathlib.Path(parent_folder)

for folder_name in os.listdir(parent_folder):
    img_count = len(list(data_dir.glob(f'{folder_name}/*')))
    print(f'{folder_name}: {img_count} images')

battery: 756 images
biological: 699 images
cardboard: 1411 images
clothes: 1892 images
glass: 1736 images
metal: 930 images
paper: 1336 images
plastic: 1597 images
shoes: 1449 images
trash: 453 images


# Train, Validation and Test Splitting

In [214]:
"""def split_dataset(parent_folder):

    # percentages
    train_ratio = 0.7
    val_ratio = 0.15
    test_ratio = 0.15

    # get base folder (Datasets/)
    base_folder = os.path.dirname(parent_folder)

    train_path = os.path.join(base_folder, "Train")
    val_path = os.path.join(base_folder, "Val")
    test_path = os.path.join(base_folder, "Test")

    # create main folders
    os.makedirs(train_path, exist_ok=True)
    os.makedirs(val_path, exist_ok=True)
    os.makedirs(test_path, exist_ok=True)

    # loop through each class
    for label in os.listdir(parent_folder):

        label_path = os.path.join(parent_folder, label)

        if not os.path.isdir(label_path):
            continue

        print("Processing:", label)

        images = os.listdir(label_path)

        # remove hidden files if any
        images = [img for img in images if not img.startswith('.')]

        random.shuffle(images)

        total = len(images)

        # calculate split counts
        train_count = int(total * train_ratio)
        val_count = int(total * val_ratio)

        # split images
        train_images = images[:train_count]
        val_images = images[train_count:train_count + val_count]
        test_images = images[train_count + val_count:]

        # create label folders
        train_label = os.path.join(train_path, label)
        val_label = os.path.join(val_path, label)
        test_label = os.path.join(test_path, label)

        os.makedirs(train_label, exist_ok=True)
        os.makedirs(val_label, exist_ok=True)
        os.makedirs(test_label, exist_ok=True)

        # copy files
        for img in train_images:
            shutil.copy(os.path.join(label_path, img), os.path.join(train_label, img))

        for img in val_images:
            shutil.copy(os.path.join(label_path, img), os.path.join(val_label, img))

        for img in test_images:
            shutil.copy(os.path.join(label_path, img), os.path.join(test_label, img))

        print(f"{label} → Total:{total} | Train:{len(train_images)}, Val:{len(val_images)}, Test:{len(test_images)}")

    print("\nDone!")"""

'def split_dataset(parent_folder):\n\n    # percentages\n    train_ratio = 0.7\n    val_ratio = 0.15\n    test_ratio = 0.15\n\n    # get base folder (Datasets/)\n    base_folder = os.path.dirname(parent_folder)\n\n    train_path = os.path.join(base_folder, "Train")\n    val_path = os.path.join(base_folder, "Val")\n    test_path = os.path.join(base_folder, "Test")\n\n    # create main folders\n    os.makedirs(train_path, exist_ok=True)\n    os.makedirs(val_path, exist_ok=True)\n    os.makedirs(test_path, exist_ok=True)\n\n    # loop through each class\n    for label in os.listdir(parent_folder):\n\n        label_path = os.path.join(parent_folder, label)\n\n        if not os.path.isdir(label_path):\n            continue\n\n        print("Processing:", label)\n\n        images = os.listdir(label_path)\n\n        # remove hidden files if any\n        images = [img for img in images if not img.startswith(\'.\')]\n\n        random.shuffle(images)\n\n        total = len(images)\n\n        # c

In [215]:
"""parent_folder = "Datasets/10_Garbage_Class"
split_dataset(parent_folder)"""

'parent_folder = "Datasets/10_Garbage_Class"\nsplit_dataset(parent_folder)'

# Data Augmentation on Train Dataset

1. Rotation and flipping of images.
2. Adjusting brightness of images.
3. Introducing color jitter.
4. Gaussian noise
5. Salt and pepper noise


In [216]:
"""train_path = 'Datasets/Train'
data_dir = pathlib.Path(train_path)

for folder_name in os.listdir(train_path):
    img_count = len(list(data_dir.glob(f'{folder_name}/*')))
    print(f'{folder_name}: {img_count} images')

total_image_count = len(list(data_dir.glob('*/*.jpg')))
print(f"Total Training Images: {total_image_count}")"""

'train_path = \'Datasets/Train\'\ndata_dir = pathlib.Path(train_path)\n\nfor folder_name in os.listdir(train_path):\n    img_count = len(list(data_dir.glob(f\'{folder_name}/*\')))\n    print(f\'{folder_name}: {img_count} images\')\n\ntotal_image_count = len(list(data_dir.glob(\'*/*.jpg\')))\nprint(f"Total Training Images: {total_image_count}")'

In [217]:
"""battery = list(data_dir.glob('battery/*'))
Image.open(str(battery[0]))"""

"battery = list(data_dir.glob('battery/*'))\nImage.open(str(battery[0]))"

In [218]:
"""train_object_image_dict = {}

for folder_name in os.listdir(train_path):
    train_object_image_dict[folder_name] = list(data_dir.glob(f"{folder_name}/*"))
"""

'train_object_image_dict = {}\n\nfor folder_name in os.listdir(train_path):\n    train_object_image_dict[folder_name] = list(data_dir.glob(f"{folder_name}/*"))\n'

In [219]:
"""object_labels_dict = {}
label = 0
for folder_name in os.listdir(train_path):
    object_labels_dict[folder_name] = label
    label += 1

object_labels_dict"""

'object_labels_dict = {}\nlabel = 0\nfor folder_name in os.listdir(train_path):\n    object_labels_dict[folder_name] = label\n    label += 1\n\nobject_labels_dict'

### Resize Images

In [220]:
"""X_train, y_train = [], []

for object_name, images in train_object_image_dict.items():
    for image in images:
        img = cv2.imread(image)
        resized_img = cv2.resize(img, (256, 256))
        X_train.append(resized_img)
        y_train.append(object_labels_dict[object_name])"""

'X_train, y_train = [], []\n\nfor object_name, images in train_object_image_dict.items():\n    for image in images:\n        img = cv2.imread(image)\n        resized_img = cv2.resize(img, (256, 256))\n        X_train.append(resized_img)\n        y_train.append(object_labels_dict[object_name])'

In [221]:
"""X_train = np.array(X_train)
y_train = np.array(y_train)"""

'X_train = np.array(X_train)\ny_train = np.array(y_train)'

### Scaling the images

In [222]:
"""X_train_scaled = X_train / 255
X_test_scaled = X_test / 255"""

'X_train_scaled = X_train / 255\nX_test_scaled = X_test / 255'

# Data Pipeline

In [223]:
train_path = "Datasets/Train"
val_path = "Datasets/Val"
IMAGE_SIZE = (256, 256)
BATCH_SIZE = 32
SEED = 123

train_ds = image_dataset_from_directory(
  train_path,
  image_size=IMAGE_SIZE,
  batch_size=BATCH_SIZE,
  seed=SEED
)

val_ds = image_dataset_from_directory(
  val_path,
  image_size=IMAGE_SIZE,
  batch_size=BATCH_SIZE,
  seed=SEED
)

class_names = train_ds.class_names

train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

train_ds = train_ds.apply(
  tf.data.experimental.ignore_errors()
)


Found 8578 files belonging to 10 classes.
Found 1833 files belonging to 10 classes.


# CNN Model Pipeline

In [224]:
num_classes = len(class_names)

cnn_params = {
    "model_name": "Convolutional Neural Network",
    "input_shape": (256, 256, 3),
    "conv1_filters": 16,
    "conv2_filters": 32,
    "conv3_filters": 64,
    "kernel_size": 3,
    "dense_units": 128,
    "optimizer": "adam",
    "loss": "SparseCategoricalCrossentropy",
    "epochs": 30,
    "early_stopping_patience": 8
}

cnn_model = Sequential([
    Rescaling(1./255, input_shape=cnn_params['input_shape']),
    Conv2D(cnn_params['conv1_filters'], cnn_params['kernel_size'], padding='same', activation='relu'),
    MaxPooling2D(),
    Conv2D(cnn_params['conv2_filters'], cnn_params['kernel_size'], padding='same', activation='relu'),
    MaxPooling2D(),
    Conv2D(cnn_params['conv3_filters'], cnn_params['kernel_size'], padding='same', activation='relu'),
    MaxPooling2D(),
    Flatten(),
    Dense(cnn_params['dense_units'], activation='relu'),
    Dense(num_classes)
])

In [225]:
# Enable GPU Memory growth 
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

In [226]:
models = [
    (
        "Convolutional Neural Network",
        cnn_model,
        train_ds,
        val_ds,
        cnn_params
    )
]

# Model Training and Tracking in Dagshub

In [227]:
dagshub.init(repo_owner='JS-Tharun', repo_name='Garbage-Image-Classifier', mlflow=True)

load_dotenv()

os.environ['MLFLOW_TRACKING_USERNAME'] = f"{os.getenv('MLFLOW_USERNAME')}"
os.environ['MLFLOW_TRACKING_PASSWORD'] = f"{os.getenv('MLFLOW_PASSWORD')}"

mlflow.set_experiment(os.environ["MLFLOW_EXPERIMENT_NAME"])
mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])

for i, element in enumerate(models):
  model_name = element[0]
  model = element[1]
  train_ds = element[2]
  val_ds = element[3]
  model_params = element[4]


  with mlflow.start_run(run_name=model_name):

    #--------------------------------------
    # Data Pipeline logging
    #--------------------------------------

    mlflow.log_param("train_data_path", train_path)
    mlflow.log_param("val_data_path", val_path)
    mlflow.log_param("Seed", SEED)
    mlflow.log_param("image_height", IMAGE_SIZE[0])
    mlflow.log_param("image_width", IMAGE_SIZE[1])
    mlflow.log_param("batch_size", BATCH_SIZE)
    mlflow.log_param("class_names", class_names)

    #--------------------------------------
    # Model training and logging
    #--------------------------------------

    model.compile(
        optimizer='adam',
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy']
    )

    early_stop = EarlyStopping(
      monitor='val_loss',
      patience=cnn_params['early_stopping_patience'],
      restore_best_weights=True
    )

    history = model.fit(
      train_ds,
      validation_data=val_ds,
      epochs=cnn_params['epochs'],
      callbacks=[early_stop]
    )

    for epoch in range(len(history.history['loss'])):
      mlflow.log_metric("train_loss", history.history['loss'][epoch], step=epoch)
      mlflow.log_metric("train_accuracy", history.history['accuracy'][epoch], step=epoch)

    mlflow.log_params(model_params)
    mlflow.tensorflow.log_model(
      model, 
      artifact_path=model_name
    )

Initialized MLflow to track repo "JS-Tharun/Garbage-Image-Classifier"

Repository JS-Tharun/Garbage-Image-Classifier initialized!

Epoch 1/30
268/268 [==============================] - 13s 47ms/step - loss: 1.7672 - accuracy: 0.3920 - val_loss: 1.5968 - val_accuracy: 0.4457
Epoch 2/30
268/268 [==============================] - 13s 48ms/step - loss: 1.2701 - accuracy: 0.5678 - val_loss: 1.3514 - val_accuracy: 0.5586
Epoch 3/30
268/268 [==============================] - 13s 47ms/step - loss: 0.9459 - accuracy: 0.6818 - val_loss: 1.3098 - val_accuracy: 0.5788
Epoch 4/30
268/268 [==============================] - 13s 48ms/step - loss: 0.6107 - accuracy: 0.7999 - val_loss: 1.4660 - val_accuracy: 0.5592
Epoch 5/30
268/268 [==============================] - 14s 50ms/step - loss: 0.3925 - accuracy: 0.8720 - val_loss: 1.9001 - val_accuracy: 0.5668
Epoch 6/30
268/268 [==============================] - 13s 49ms/step - loss: 0.2323 - accuracy: 0.9271 - val_loss: 2.1940 - val_accuracy: 0.5374
Epoch 7/30
268/268 [==============================] - 13s 48ms/step - loss: 0.1963 - accuracy: 0.9386 - val_loss: 2.2716 - val_accuracy:

2026/04/17 21:26:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/17 21:26:09 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


INFO:tensorflow:Assets written to: C:\Users\jstha\AppData\Local\Temp\tmp1qg31i2a\model\data\model\assets


INFO:tensorflow:Assets written to: C:\Users\jstha\AppData\Local\Temp\tmp1qg31i2a\model\data\model\assets


🏃 View run Convolutional Neural Network at: https://dagshub.com/JS-Tharun/Garbage-Image-Classifier.mlflow/#/experiments/1/runs/9d08d0bc9157409e8b4d0be26bc53cba
🧪 View experiment at: https://dagshub.com/JS-Tharun/Garbage-Image-Classifier.mlflow/#/experiments/1
